In [3]:
## Import Libraries
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import CTransformers

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore


## Create Pinecode Cluster

In [4]:
pinecone_api_key ='pcsk_3GtRpL_LvnaxPMgB3U2UcH8KjZtC2i8GJduoRPNDQnXWd429kNciXRwLgXjFKAttseSLmt'


## 1. Load the data

Extract data from the PDF book

In [5]:
def load_pdf(data):
    loader=DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    docs = loader.load()
    return docs


In [6]:
extracted_data = load_pdf('data/')

## 2. Convert Corpus to Text Chunks

In [7]:
# Create text splitter
def text_splitter(extracted_datadata):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
        #separators=["\n\n", "\n", " ", ""]
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [8]:
text_chunks = text_splitter(extracted_data)

In [8]:
len(text_chunks)

7024

## 3. Convert Chunks into Vectors

In [9]:
# Download embedding model and create embeddings 
def download_embedding_model():  
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embedding_model

In [10]:
import sentence_transformers
embeddings = download_embedding_model()

c:\Users\MRAdmin\miniconda3\envs\mchatbot\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MRAdmin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falli

## 4. Initializing Pinecone     

In [11]:
import os
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY", pinecone_api_key)  # keep your fallback
INDEX_NAME = "medical-chatbot"

pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index if it doesn't exist (Serverless)
# IMPORTANT: dimension must match your embedding model (MiniLM-L6-v2 => 384)
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(INDEX_NAME)

# Create LangChain vector store wrapper
vectorstore = PineconeVectorStore(index=index, embedding=embeddings)

# Add your documents (upsert)
vectorstore.add_documents(text_chunks)


['43b162ed-207a-4cf8-9a77-aa1e703987f4',
 'd8c1f27b-d25c-4339-a78a-af5fcecdc3c6',
 '4dc79779-fc64-4438-b426-03dd8511c760',
 '86d5c8cd-3fab-4b6b-bb1f-426d4e3af78a',
 'e6be72f3-82a8-4122-a3dd-f78a513a853f',
 'c1bf80ee-5c71-4473-844d-373f42015604',
 'fc46ea0f-938d-4726-9058-f9b27027d699',
 '71b7bc12-42bd-4513-8b35-524aa8dc5946',
 'e2e5f3e6-c03e-407b-81a2-149e3bafabff',
 'acf6a9d0-b89c-4680-bcf8-5bb8e70f46b9',
 'f393da4d-677c-4a89-8ec7-766061f42c03',
 '20705d45-9e92-40c1-ab52-af2a9ce395d9',
 '512cd8f0-17c3-42b1-8222-a1c780d0d948',
 '21686b33-e2f7-494e-9926-a95014979f92',
 '4d5c80f9-c6d8-48a4-9432-274bbb4c8d56',
 '1626263b-d29f-4e01-97e0-b61251c80148',
 'b78e3ba6-cbd1-4ba4-9e89-f23f86804c5d',
 'd9675c3a-7a5b-45c5-98de-e1c6dc6a6736',
 '3ede1217-4af8-4848-9893-e0440caa8442',
 '3f923906-54a3-4235-95e1-379efb80229e',
 '1dd8b55d-1bec-43ba-93fc-f53afe8ddd1c',
 '9d5563f0-f303-4c19-bac6-376b3399e0de',
 '9a17abc6-f68f-4bb4-b3fd-63b433a75462',
 '3fe0ebec-fcbd-48c5-b08b-28f4964dda25',
 '30713217-5369-

## 5. Build retrieval QA chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.llms import CTransformers

# Prompt (modern)
prompt = ChatPromptTemplate.from_template(
    "Use the following context to answer the question.\n\n"
    "Context:\n{context}\n\n"
    "Question:\n{question}\n\n"
    "Answer:"
)

# LLM
llm = CTransformers(
    model="model/llama-2-7b-chat.ggmlv3.q4_0.bin",
    model_type="llama",
    config={"max_new_tokens": 256, "temperature": 0.1},
)

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Helper to convert retrieved Documents -> single context string
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# Retrieval + prompt + LLM chain
qa = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Example query:
print(qa.invoke("What is this document set about?"))



 This document is a copyright notice for a publication, indicating that it is protected by copyright law and that any unauthorized use or reproduction will be vigorously defended. The document also provides information about the publisher, including their name, address, and contact details.


In [14]:
print(qa.invoke("Summarize the symptoms mentioned in the documents."))

Number of tokens (513) exceeded maximum context length (512).
Number of tokens (514) exceeded maximum context length (512).
Number of tokens (515) exceeded maximum context length (512).
Number of tokens (516) exceeded maximum context length (512).
Number of tokens (517) exceeded maximum context length (512).
Number of tokens (518) exceeded maximum context length (512).
Number of tokens (519) exceeded maximum context length (512).
Number of tokens (520) exceeded maximum context length (512).
Number of tokens (521) exceeded maximum context length (512).
Number of tokens (522) exceeded maximum context length (512).
Number of tokens (523) exceeded maximum context length (512).
Number of tokens (524) exceeded maximum context length (512).
Number of tokens (525) exceeded maximum context length (512).
Number of tokens (526) exceeded maximum context length (512).
Number of tokens (527) exceeded maximum context length (512).
Number of tokens (528) exceeded maximum context length (512).
Number o


The symptoms mentioned in the document are:
• Loss of short-term memory or ability to concentrate
• Sore throat
• Tender lymph nodes
• Muscle pain
• Multi-joint pain without swelling or redness
• Headaches of a new type, pattern, or severity
• Unrefreshing sleep
• Post-exertional malaise (a vague feeling of discomfort or tiredness following exertion)
• Eagerness to undergo operations and other procedures
• Arguments with hospital staff or similar acting-out behaviors
• Few visitors
• Ongoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoing to undergoin